# Misinformation_detection.ipynb

**Author:** Zane Zhang

**Date:** 2025/06/23 20:51

**Description:**

This script aims to detect depression misinformation using LLMs based on Depression Misinformation Dataset. The notebook is better to run with a cloud service (Colab Pro+ here) because of its demanding memory and GPU.

First, we randomly select 5 shots from DMisinfo to train our LLM-based models.

Second, we apply In-context Learning to all models and build a classifier to label information. In this step we apply *MentalLLaMa* to achieve our goal.

*MentalLLaMa* model can be seen at <https://github.com/SteveKGYang/MentalLLaMA?tab=readme-ov-file>.

**Note**: This script is ultimately not used because of the maximum length limit for input text.

In [ ]:
!pip install transformers accelerate sentencepiece

In [ ]:
import pandas as pd
import torch
from google.colab import drive
drive.mount('/content/drive')
from transformers import LlamaTokenizer, LlamaForCausalLM

First, we randomly select 5 shots (3 DMisinfo, 2 non-DMisinfo) from DMisinfo. We use the same 5 shots to train all LLM-based models.

In [ ]:
df = pd.read_excel("drive/MyDrive/DMisinfo/manual_annotation.xlsx")
sample_1 = df[df['Label'] == 1].sample(n=3, random_state=42)
sample_0 = df[df['Label'] == 0].sample(n=2, random_state=42)
sample = pd.concat([sample_1, sample_0]).sample(frac=1, random_state=42)    # disorder

titles = sample['Title_normalized'].fillna('').tolist()
bodies = sample['Sentence_normalized'].fillna('').tolist()
answers = sample['Answer'].tolist()

Then we build a prompt template and function to detect misinformation.

In [ ]:
# Prompt input template for detecting DMisinfo using LLMs
examples = ""
for i, (title, body, answer) in enumerate(zip(titles, bodies, answers), 1):
    examples += f"Example {i}: {title} {body}\nAnswer: {answer}\n"

prompt_prefix = f"""You are an expert psychiatrist who has comprehensive knowledge on depression conditions and misinformation surrounding it. You are helpful,so you will try your best to give accurate answers. Now, thoroughly look at the following examples which contains the post and and rationale on whether a post contains misinformation regarding depression or not.
{examples}
Now, only using the post data provided below, answer whether the post contains depression misinformation or not. You must include the word 'yes' or 'no' within your answer, then give your reasoning.
"""

# Save examples
with open("drive/MyDrive/DMisinfo/examples.txt", "w", encoding='utf-8') as f:
    f.write(examples)

Before we use the MentalLLaMa model to detect depression misinformation, we should install and load it. `pip install transformers accelerate bitsandbytes` and load like following.

In the cell, `device_map="auto"` means using GPU automatically if available, `torch_dtype=torch.float16` sets the precision to save memory and speed up inference. `model.eval()` means setting the model mode to evaluation to keep stable and efficient.

In [ ]:
# Load model
model_path = 'klyang/MentaLLaMA-chat-13B'

tokenizer = LlamaTokenizer.from_pretrained(model_path)
model = LlamaForCausalLM.from_pretrained(
    model_path,
    device_map="auto",  # auto select CPU/GPU
    torch_dtype=torch.float16 # half precision to save memory
)
model.eval()

**Problem Formulation** We define the detection of depression-related misinformation in Reddit posts as a binary classification problem. Given a post D = (T, B) where T is the title and B is the normalized body text, we aim to learn a classifier F(T, B) → y, where y ∈ {0, 1} represents whether the post contains depression-related mental health misinformation.

The function `classify_post` takes a post's title and body as input and uses the loaded MentalLLaMa model to determine if it contains depression misinformation. `max_tokens` is an optional parameter for controlling the length of the model's response. `return_tensors="pt"` specifies that the output should be PyTorch tensors, and `.to(model.device)`moves these tensors to the same device as the model.

`with torch.no_grad()` is a context manager that disables gradient calculation. This is important during inference (when not training the model) as it reduces memory usage and speed up computation. `generate_ids = model.generate(...)` generates a response from the model based on the input.

- `inputs.input_ids` provides the tokenized input.
- `max_new_tokens=max_tokens` limits the maximum number of new tokens the model will generate in its response.
- `do_sample=False` disables sampling, making the output deterministic (the model will produce the same output for the same input).
- `temperature=0.3` controls the radnomness of the output when sampling is enabled (not active here). A lower temperature makes the output more focused and less creative.
- `eos_token_id=tokenizer.eos_token_id` specifies the token ID that signals the end of the generation.

`response = tokenizer.batch_decode(...)` decodes the generated token IDs back into a human-readable string. `skip_special_tokens=True` removes any special tokens used by the tokenizer, and `clean_up_tokenization_spaces=True` cleans up extra spaces. `[0]` is used to get the first (and in this case, only) generated sequence. `return response.replace(full_prompt, "").strip()` removes the original prompt from the generated response and leading/trailing whitespace, returning only the model's generated answer.

In [ ]:
# Classifier
def classify_post(title: str, body: str, max_tokens=128):
    prompt = prompt_prefix + f"\nPost: {title} {body}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generate_ids = model.generate(
            inputs.input_ids,
            max_new_tokens=max_tokens,
            do_sample=False,
            # temperature=0.3,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)[0]
    return response.replace(prompt, "").strip()

Next step is local detection. We first prepare unlabeled data, then label each term within it. If one term is regarded as misinformation, it should be labeled as 1; otherwise, it should be labeled as 0

In [ ]:
# Prepare unlabeled data
df = pd.read_csv("drive/MyDrive/DMisinfo/posts_en.csv")
manual_df = pd.read_excel("drive/MyDrive/DMisinfo/manual_annotation.xlsx")
manual_ids = set(manual_df['Post_id'])
df = df[~df['Post_id'].isin(manual_ids)].reset_index(drop=True)

# Classify posts
results = []
total = len(df)
for idx, row in enumerate(df.itertuples(), 1):
    output = classify_post(row.Title_normalized, row.Sentence_normalized)
    label = 1 if "yes" in output.lower() else 0 if "no" in output.lower() else -1
    results.append({
        "id": row.Post_id,
        "title": row.Title_normalized,
        "body": row.Sentence_normalized,
        "output": output,
        "label": label
    })

    if idx % 100 == 0 or idx == total:
        print(f"Processed {idx}/{total} posts")

# Save results
pd.DataFrame(results).to_csv("drive/MyDrive/DMisinfo/results_mental_llama.csv", index=False, encoding='utf-8')